# AC Stark Discriminator: Residual Photons vs Intrinsic Detuning

The ghost-frequency hypothesis (quasi-static 20–40 kHz detuning) survived
obvious mundane explanations. Here we test a sharper first-principles candidate:

**AC Stark shift from residual resonator photons.**
A readout resonator with mean photon number n̄ Stark-shifts its qubit by
δ = 2χn̄. For χ/2π ≈ 1 MHz and n̄ ≈ 0.015, δ ≈ 30 kHz — right in the
observed band. This matches ALL ghost signatures: spectator-independent,
branch-independent, invisible to Hahn-echo T2.

**Discriminator:** resonator photons decay (T1r ~ µs); intrinsic detuning does not.
Insert a variable pre-delay τ between init and the Ramsey sequence:
- Photonic → extracted detuning DECAYS as exp(−τ/T1r)
- Intrinsic → extracted detuning stays CONSTANT

If the QPU shows decaying detuning, the 'ghost' is photons — a known effect,
not new physics. If constant, the ghost survives its hardest test yet.


In [ ]:
import torch, numpy as np, math
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))
torch.manual_seed(1)


## Physical model

Resonator photon number after pre-delay τ: n̄(τ) = n̄₀ · exp(−τ/T1r).
Instantaneous Stark detuning: δ(τ) = 2χ · n̄(τ).
Ramsey (idle t after pre-delay τ) accumulates phase:
φ(τ,t) = 2π ∫₀ᵗ 2χ·n̄₀·e^{−(τ+t′)/T1r} dt′.
We extract an *effective* detuning δ_eff(τ) = φ(τ,t)/(2πt) at fixed t.


In [ ]:
# Parameters (representative Heron values)
chi_khz = 1000.0      # chi/2pi in kHz (qubit-resonator dispersive shift)
nbar0 = 0.015         # residual photon number after readout
T1r_us = 5.0          # resonator photon decay time
t_ramsey_us = 4.4     # Ramsey idle (matches Kingston observation)
intrinsic_khz = 30.0  # hypothesized intrinsic detuning (comparison)

delta0_khz = 2*chi_khz*nbar0
print(f'Stark detuning at tau=0: {delta0_khz:.1f} kHz (observed band: 20-40 kHz)')

def stark_phase(tau_us, t_us, nbar0=nbar0, T1r_us=T1r_us, chi_khz=chi_khz):
    # analytic integral of decaying Stark shift over Ramsey idle
    tau, t, T1r = tau_us*1e-6, t_us*1e-6, T1r_us*1e-6
    integral = 2*chi_khz*1e3*nbar0*T1r*(math.exp(-tau/T1r)-math.exp(-(tau+t)/T1r))
    return 2*math.pi*integral

def effective_detuning_khz(tau_us, model='photonic'):
    if model == 'photonic':
        ph = stark_phase(tau_us, t_ramsey_us)
    else:
        ph = 2*math.pi*intrinsic_khz*1e3*t_ramsey_us*1e-6
    return ph/(2*math.pi*t_ramsey_us*1e-6)/1e3


## Discriminator: effective detuning vs pre-delay
Simulate the experiment: for each pre-delay τ, 'measure' the Ramsey phase
(with shot noise on GPU) and extract δ_eff.


In [ ]:
pre_delays = np.array([0., 2., 5., 10., 20., 40.])
n_shots = 2000

def measure_delta_eff(tau_us, model, n_shots=n_shots):
    d_true = effective_detuning_khz(tau_us, model)
    # Ramsey P(1) at fixed idle; phase from population with shot noise
    p_true = 0.5*(1+math.cos(2*math.pi*d_true*1e3*t_ramsey_us*1e-6))
    shots = torch.rand(n_shots, device=device) < p_true
    p_hat = shots.float().mean().item()
    c = max(-1.0, min(1.0, 2*p_hat-1))
    phi_hat = math.acos(c)
    return phi_hat/(2*math.pi*t_ramsey_us*1e-6)/1e3

print(f"{'tau (us)':>9} {'photonic (kHz)':>15} {'intrinsic (kHz)':>16}")
for tau in pre_delays:
    dp = measure_delta_eff(tau, 'photonic')
    di = measure_delta_eff(tau, 'intrinsic')
    print(f"{tau:>9.1f} {dp:>15.1f} {di:>16.1f}")


## Verdict

Read the table above:
- **Photonic column decays** toward zero with τ (timescale ~T1r).
  → If the QPU shows this, the ghost is residual photons. Known physics.
  → Mitigation: active resonator reset before Ramsey; detuning should vanish.
- **Intrinsic column stays flat** at 30 kHz.
  → If the QPU shows this, the ghost survives its hardest falsification yet,
    and the full-chip checkerboard campaign is justified.

Either outcome is a win: we learn whether to chase photons or new physics.
This is what 'falsifying hard enough' looks like.
